In [ ]:
import torch
import pandas as pd
import joblib
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("dataset.csv").reset_index()
LABELS = len(df['label_tec'].value_counts())
encoder = LabelEncoder()
df['enc_label'] = encoder.fit_transform(df['label_tec'])


In [ ]:
MAX_LEN = 512
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-5


In [ ]:
# CTI-BERT model and tokenizer
MODEL_NAME = "ibm-research/CTI-BERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, output_hidden_states=True)

In [ ]:
# DataLoader
class Triage(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = str(self.data.sentence[index])
        inputs = self.tokenizer.encode_plus(
            sentence, None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False
        )
        return {
            'ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'mask': torch.tensor(inputs['attention_mask'], dtype=torch.long),
            'targets': torch.tensor(self.data.enc_label[index], dtype=torch.long)
        }

    def __len__(self):
        return len(self.data)

In [ ]:
# Train-test split
train_dataset, test_dataset = train_test_split(df, test_size=0.2, stratify=df['enc_label'], random_state=42)
train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)
training_set = Triage(train_dataset, tokenizer, MAX_LEN)
testing_set = Triage(test_dataset, tokenizer, MAX_LEN)

training_loader = DataLoader(training_set, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
testing_loader = DataLoader(testing_set, batch_size=VALID_BATCH_SIZE)

In [ ]:
# CTI-BERT classifier
class CTIBERTClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(config.hidden_size, config.hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(config.hidden_size // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        pooled = torch.sum(hidden * attention_mask.unsqueeze(-1), dim=1) / attention_mask.sum(1, keepdim=True)
        return self.classifier(self.dropout(pooled))

        # I had sought AI help to build this function

In [ ]:
# Model initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CTIBERTClassifier(MODEL_NAME, LABELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
# Training loop
def train(epoch):
    model.train()
    total_loss = 0
    for batch in training_loader:
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        targets = batch['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(ids, mask)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(training_loader):.4f}")

for epoch in range(EPOCHS):
    train(epoch)

In [ ]:

def check_accuracy(loader, model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            targets = batch['targets'].to(device)
            outputs = model(ids, mask)
            _, preds = torch.max(outputs, dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    print(f"Got {correct} / {total} correct with accuracy {100 * correct / total:.2f}%")

check_accuracy(testing_loader, model)

In [ ]:
# Saving artifacts
os.makedirs("mitre_model_ctibert", exist_ok=True)
torch.save(model.state_dict(), "mitre_model_ctibert/ctibert_model.pt")
tokenizer.save_pretrained("mitre_model_ctibert")
joblib.dump(encoder, "mitre_model_ctibert/label_encoder.pkl")

# Zip & download folder
import shutil
from google.colab import files
shutil.make_archive("mitre_model_ctibert", 'zip', "mitre_model_ctibert")
files.download("mitre_model_ctibert.zip")
